In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
import mujoco_playground
from mujoco_playground._src import mjx_env
from mujoco_playground import wrapper
from ml_collections import config_dict
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import imageio.v2 as imageio
from IPython import display as ipython_display

from rlx.playground.ppo import PPOConfig, train, get_obs, normalize_obs

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────

_PLAYGROUND_PATH  = os.path.dirname(mujoco_playground.__file__)
_OP3_ASSETS_PATH  = os.path.join(_PLAYGROUND_PATH, "external_deps", "mujoco_menagerie", "robotis_op3", "assets")
_OP3_XML_PATH     = os.path.join(_PLAYGROUND_PATH, "_src", "locomotion", "op3", "xmls", "op3_mjx_feetonly.xml")

# ── Ball params ───────────────────────────────────────────────────────────────

BALL_SIZE   = 0.04
BALL_HEIGHT = 0.5
BALL_X      = 0.12
BALL_Y      = 0.05

# ── Build XML ─────────────────────────────────────────────────────────────────

with open(_OP3_XML_PATH) as f:
    _xml = f.read()

_xml = _xml.replace(
    'meshdir="../../../../mujoco_menagerie/robotis_op3/assets"',
    f'meshdir="{_OP3_ASSETS_PATH}"'
)

_VISUAL = """
  <visual>
    <map force="0.1" zfar="30" znear="0.1"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global offwidth="1280" offheight="720" elevation="-20" azimuth="120"/>
    <quality shadowsize="2048" offsamples="4"/>
  </visual>
  <statistic center="0 0 0.3" extent="1.5"/>"""

_EXTRA_ASSETS = """
    <texture type="skybox" builtin="gradient" rgb1=".3 .5 .7" rgb2="0 0 0" width="32" height="512"/>
    <texture name="texplane" type="2d" builtin="checker" rgb1=".4 .4 .4" rgb2=".6 .6 .6"
             width="512" height="512"/>
    <material name="MatPlane" reflectance="0.3" texture="texplane" texrepeat="1 1"
              texuniform="true" rgba=".7 .7 .7 1"/>"""

_FLOOR = """
    <light directional="true" diffuse=".8 .8 .8" pos="0 0 10" dir="0 0 -10"/>
    <geom name="floor" size="5 5 .5" type="plane" material="MatPlane" contype="1" conaffinity="1"/>"""

_BALL_BODY = f"""
    <body name="ball" pos="{BALL_X} {BALL_Y} {BALL_HEIGHT}">
      <freejoint name="ball_joint"/>
      <geom name="ball" type="sphere" size="{BALL_SIZE}" mass="0.045"
            friction="0.7 0.075 0.075" solref="0.02 1.0"
            rgba="0.1 0.9 0.1 1" contype="1" conaffinity="1"/>
    </body>"""

_CONTACT = """
  <contact>
    <pair geom1="l_foot1" geom2="ball"/>
    <pair geom1="l_foot2" geom2="ball"/>
  </contact>"""

_xml = _xml.replace(
    '<compiler',
    _VISUAL + "\n\n  <compiler"
)
_xml = _xml.replace("</asset>", _EXTRA_ASSETS + "\n  </asset>")
_xml = _xml.replace("<worldbody>", "<worldbody>" + _FLOOR)
_xml = _xml.replace("</worldbody>", _BALL_BODY + "\n  </worldbody>")
_XML = _xml.replace("</mujoco>", _CONTACT + "\n</mujoco>")

# ── Helpers ───────────────────────────────────────────────────────────────────

def _mjx_step(mjx_model, data, action, n_substeps):
    """Apply action for n_substeps physics steps."""
    def single_step(data, _):
        data = data.replace(ctrl=action)
        return mjx.step(mjx_model, data), None
    return jax.lax.scan(single_step, data, (), n_substeps)[0]

# ── Environment ───────────────────────────────────────────────────────────────

def _robot_tricks_config():
    return config_dict.create(
        ctrl_dt=0.02,    # 5 substeps × 0.004 s
        sim_dt=0.004,
        impl="jax",
        nconmax=200,
        njmax=200,
    )


class RobotTricksEnv(mjx_env.MjxEnv):
    """OP3 robot ball-bouncing environment for mujoco_playground PPO."""

    def __init__(self, config=None, config_overrides=None):
        super().__init__(config or _robot_tricks_config(), config_overrides)
        self._mj_model = mujoco.MjModel.from_xml_string(_XML)
        self._mj_model.opt.solver    = mujoco.mjtSolver.mjSOL_CG
        self._mj_model.opt.iterations    = 6
        self._mj_model.opt.ls_iterations = 6
        self._mj_model.opt.timestep  = self.sim_dt
        self._mjx_model = mjx.put_model(self._mj_model, impl=self._config.impl)
        self._foot_left_id = self._mj_model.body("l_ank_roll_link").id   # body 15
        self._ball_id      = self._mj_model.body("ball").id              # body 22
        # ball freejoint starts at qpos[27] → ball_z = qpos[29]
        self._ball_qpos_start = int(
            self._mj_model.jnt_qposadr[
                self._mj_model.body_jntadr[self._ball_id]
            ]
        )

    def reset(self, rng: jax.Array) -> mjx_env.State:
        rng, rng1, rng2 = jax.random.split(rng, 3)
        data = mjx.make_data(self.mj_model, impl=self._config.impl)
        # Add small noise to OP3 DOFs only (skip ball freejoint)
        n_op3_q = self.mj_model.nq - 7          # 27 (op3 freejoint + joints)
        n_op3_v = self.mj_model.nv - 6          # 26
        noise_scale = 1e-2
        noise_q = jax.random.uniform(rng1, (n_op3_q,), minval=-noise_scale, maxval=noise_scale)
        noise_v = jax.random.uniform(rng2, (n_op3_v,), minval=-noise_scale, maxval=noise_scale)
        qpos = data.qpos.at[:n_op3_q].add(noise_q)
        qvel = data.qvel.at[:n_op3_v].set(noise_v)
        data = data.replace(qpos=qpos, qvel=qvel)
        data = mjx.forward(self.mjx_model, data)

        ball_z = data.qpos[self._ball_qpos_start + 2]
        metrics = {
            "reward/ball":  jnp.zeros(()),
            "reward/ctrl":  jnp.zeros(()),
            "reward/alive": jnp.zeros(()),
            "bounces":      jnp.zeros(()),
        }
        info = {"rng": rng, "prev_ball_z": ball_z}
        obs  = self._get_obs(data, info)
        reward, done = jnp.zeros(2)
        return mjx_env.State(data, obs, reward, done, metrics, info)

    def step(self, state: mjx_env.State, action: jax.Array) -> mjx_env.State:
        data = _mjx_step(self.mjx_model, state.data, action, self.n_substeps)

        bqs      = self._ball_qpos_start
        ball_pos = data.qpos[bqs:bqs + 3]          # ball xyz
        ball_z   = ball_pos[2]
        foot_pos = data.xpos[self._foot_left_id]    # left foot world pos
        torso_z  = data.qpos[2]                     # op3 freejoint z

        # Alive: torso upright (OP3 standing height ~0.3 m)
        alive = (
            (torso_z >= 0.2) & (torso_z <= 1.5)
        ).astype(jnp.float32)
        dist_foot_ball = jnp.sqrt(jnp.sum(jnp.square(ball_pos[:2] - foot_pos[:2])))
        alive = jnp.where(dist_foot_ball > 1.5, 0.0, alive)

        ctrl_cost = 0.1  * jnp.sum(jnp.square(action))
        ball_rwd  = 3.0  * (1.0 - jnp.abs(ball_z - BALL_HEIGHT) / 1.5) * (ball_z >= 0.05)
        foot_rwd  = 3.0  * jnp.clip(1.0 - dist_foot_ball / 1.5, 0.0, 1.0)
        alive_rwd = 3.0
        reward    = ball_rwd + foot_rwd + alive_rwd - ctrl_cost

        bounce_thr = BALL_HEIGHT - 0.05
        is_bounce  = (
            (state.info["prev_ball_z"] < bounce_thr) & (ball_z >= bounce_thr)
        ).astype(jnp.float32)

        metrics = {
            "reward/ball":  ball_rwd,
            "reward/ctrl":  -ctrl_cost,
            "reward/alive": alive_rwd * alive,
            "bounces":      is_bounce,
        }
        info = {**state.info, "prev_ball_z": ball_z}
        obs  = self._get_obs(data, info)
        done = 1.0 - alive
        return mjx_env.State(data, obs, reward, done, metrics, info)

    def _get_obs(self, data: mjx.Data, info: dict) -> dict:
        # qpos[2:]: position excl. global x,y  (32 dims)
        # qvel:     velocities                  (32 dims)
        # xpos[1:]: world-frame body positions  (22 × 3 = 66 dims)
        # xmat[1:]: world-frame body rotations  (22 × 9 = 198 dims)
        # qfrc_actuator: actuator forces         (32 dims)
        obs = jnp.concatenate([
            data.qpos[2:],
            data.qvel,
            data.xpos[1:].ravel(),
            data.xmat[1:].ravel(),
            data.qfrc_actuator,
        ])
        return {"state": obs, "privileged_state": obs}

    @property
    def xml_path(self) -> str:
        return _OP3_XML_PATH

    @property
    def action_size(self) -> int:
        return self._mjx_model.nu

    @property
    def mj_model(self) -> mujoco.MjModel:
        return self._mj_model

    @property
    def mjx_model(self) -> mjx.Model:
        return self._mjx_model


# Smoke-test
env = RobotTricksEnv()
s0  = jax.jit(env.reset)(jax.random.key(0))
s1  = jax.jit(env.step)(s0, jnp.zeros(env.action_size))
print(f"obs shape: {s0.obs['state'].shape}   action_size: {env.action_size}")

In [ ]:
cfg = PPOConfig(
    env_id="RobotTricks",
    num_envs=2048,
    total_timesteps=100_000_000,
    num_steps=10,
    num_minibatches=32,
    update_epochs=8,
    learning_rate=3e-4,
    gamma=0.9,
    gae_lambda=0.95,
    clip_coef=0.3,
    ent_coef=1e-3,
    vf_coef=0.5,
    max_grad_norm=1.0,
    norm_adv=True,
    actor_hidden_sizes=[32, 32, 32, 32],
    critic_hidden_sizes=[256, 256, 256, 256, 256],
    policy_obs_key="state",
    value_obs_key="privileged_state",
    log_frequency=10,
    eval_frequency=200,
    eval_episodes=128,
    use_checkpointing=False,
    use_wandb=False,
    verbose=False,
    progress_bar=False,
    seed=42,
)

# ── Live plot ─────────────────────────────────────────────────────────────────

PLOT_KEYS = [
    # row 1 — task performance
    ("training/episode_reward", "Episode reward",   "C0"),
    ("eval/bounces",            "Bounces (eval)",   "C1"),
    ("time/sps",                "Steps / sec",      "C5"),
    # row 2 — PPO losses
    ("loss/policy",             "Policy loss",      "C2"),
    ("loss/value",              "Value loss",       "C3"),
    ("diagnostics/clip_frac",   "Clip fraction",    "C4"),
    # row 3 — sub-rewards
    ("eval/reward/ball",        "Ball reward",      "C6"),
    ("eval/reward/alive",       "Alive reward",     "C7"),
    ("eval/reward/ctrl",        "Ctrl cost",        "C8"),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
axes = axes.ravel()
for ax, (_, label, _c) in zip(axes, PLOT_KEYS):
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("steps (M)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
fig.tight_layout()

logs = []

def log_callback(log, step):
    logs.append({"step": step, **log})
    if len(logs) < 2:
        return
    steps_m = [d["step"] / 1e6 for d in logs]
    for ax, (key, label, color) in zip(axes, PLOT_KEYS):
        vals  = [d.get(key) for d in logs]
        valid = [(s, v) for s, v in zip(steps_m, vals) if v is not None]
        if len(valid) < 2:
            continue
        xs, ys = zip(*valid)
        ax.cla()
        ax.plot(xs, ys, color=color, linewidth=1.2)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel("steps (M)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"RobotTricks — {step/1e6:.1f} M / {cfg.total_timesteps/1e6:.0f} M steps", fontsize=11)
    fig.tight_layout()
    ipython_display.clear_output(wait=True)
    ipython_display.display(fig)

# ── Train ─────────────────────────────────────────────────────────────────────

model, optimizer, actor_stats, critic_stats = train(
    cfg,
    env_factory=RobotTricksEnv,
    log_callback=log_callback,
)

# Final: save and close (prevents Jupyter from auto-displaying a second time)
plt.savefig("robot_tricks_training.png", dpi=120)
plt.close(fig)

In [ ]:
# ── Rollout with trained policy ───────────────────────────────────────────────

raw_env   = RobotTricksEnv()
jit_reset = jax.jit(raw_env.reset)
jit_step  = jax.jit(raw_env.step)

@jax.jit
def policy_fn(obs, actor_stats):
    norm_a = normalize_obs(get_obs(obs, cfg.policy_obs_key), actor_stats)
    norm_c = normalize_obs(get_obs(obs, cfg.value_obs_key), critic_stats)
    dist, _ = model(norm_a, norm_c)
    return dist.mode  # deterministic

key   = jax.random.key(1)
state = jit_reset(key)
trajectory = [state]

for _ in range(1000):
    action = policy_fn(state.obs, actor_stats)
    state  = jit_step(state, action)
    trajectory.append(state)
    if float(state.done) > 0.5:
        break

total_bounces = sum(float(s.metrics["bounces"]) for s in trajectory)
total_reward  = sum(float(s.reward) for s in trajectory)
print(f"Episode length: {len(trajectory)} steps")
print(f"Total reward:   {total_reward:.1f}")
print(f"Ball bounces:   {total_bounces:.0f}")

# ── Render GIF ────────────────────────────────────────────────────────────────

render_every = 2
frames = raw_env.render(
    trajectory[::render_every], camera="track", height=480, width=640
)
fps = int(1.0 / raw_env.dt / render_every)

gif_path = "robot_tricks.gif"
imageio.mimsave(
    gif_path,
    [np.uint8(f) for f in frames],
    fps=fps,
    loop=0,
)
print(f"Saved → {gif_path}  ({len(frames)} frames @ {fps} fps)")

from IPython.display import Image
Image(gif_path)